# 💱 Dashboard SQL Interactivo de Tipos de Cambio

**Autor:** Enrique H.G. 

Este notebook consulta datos dinámicos desde una **API pública** (Frankfurter — tipos de cambio
históricos del Banco Central Europeo, sin necesidad de API key), los carga en una **base de
datos SQLite en memoria** y responde a los controles interactivos ejecutando **consultas SQL
reales** (no operaciones de pandas) directamente sobre la base de datos:

- Series de tiempo simples (`SELECT ... WHERE ... ORDER BY`)
- Medias móviles calculadas con **funciones de ventana** (`AVG() OVER (... ROWS BETWEEN ...)`)
- Volatilidad diaria con `LAG()` para variación porcentual día a día

La idea del notebook es mostrar manejo de **SQL dentro de Jupyter** (motor `sqlite3`, estándar
de la librería de Python) combinado con una fuente de datos dinámica y visualización
interactiva — sin depender de Streamlit ni Dash.

> Ejecutar las celdas en orden. La conexión y la base de datos persisten como variables
> globales, por lo que el panel se mantiene interactivo tras la última celda.

In [1]:
# Importacion de librerias y configuracion del entorno
import warnings
warnings.filterwarnings("ignore")

import sqlite3
import requests
from datetime import date, timedelta
import pandas as pd
import numpy as np

import plotly.graph_objects as go
import plotly.io as pio

import ipywidgets as widgets
from IPython.display import display, Markdown, clear_output

# Fuerza el renderizado de graficos Plotly dentro de las celdas del notebook
pio.renderers.default = "notebook"

print(f"Librerias cargadas. Motor SQL: sqlite3 (version {sqlite3.sqlite_version}), "
      f"soporta funciones de ventana desde 3.25.0")


Librerias cargadas. Motor SQL: sqlite3 (version 3.51.0), soporta funciones de ventana desde 3.25.0


In [2]:
# Base de datos SQLite en memoria -- esquema y conexion persistente

CONEXION_SQL = sqlite3.connect(":memory:")

CONEXION_SQL.execute("""
CREATE TABLE tipos_cambio (
    fecha           TEXT NOT NULL,
    moneda_base     TEXT NOT NULL,
    moneda_destino  TEXT NOT NULL,
    tasa            REAL NOT NULL,
    PRIMARY KEY (fecha, moneda_base, moneda_destino)
)
""")
CONEXION_SQL.commit()

# Registro de combinaciones (base, destinos, rango) ya cargadas -- evita re-consultar la API
_COMBINACIONES_CARGADAS = set()

print("Base de datos SQLite en memoria creada (tabla 'tipos_cambio').")


Base de datos SQLite en memoria creada (tabla 'tipos_cambio').


In [3]:
# Capa de datos -- consulta a la API publica de Frankfurter e insercion via SQL

URL_BASE_API = "https://api.frankfurter.dev/v1"


def cargar_tipos_cambio(moneda_base, monedas_destino, fecha_inicio, fecha_fin):
    """Descarga tasas historicas desde la API publica y las inserta en SQLite (INSERT OR IGNORE).

    Devuelve (True, None) si tiene exito, o (False, mensaje_error) si falla.
    """
    clave = (moneda_base, tuple(sorted(monedas_destino)), fecha_inicio, fecha_fin)
    if clave in _COMBINACIONES_CARGADAS:
        return True, None

    try:
        simbolos = ",".join(monedas_destino)
        url = f"{URL_BASE_API}/{fecha_inicio}..{fecha_fin}"
        respuesta = requests.get(
            url, params={"base": moneda_base, "symbols": simbolos}, timeout=10
        )
        respuesta.raise_for_status()
        datos_json = respuesta.json()

        tasas_por_fecha = datos_json.get("rates", {})
        if not tasas_por_fecha:
            raise ValueError("La API respondio sin tasas para este rango de fechas/monedas.")

        registros = [
            (fecha, moneda_base, moneda_destino, tasa)
            for fecha, tasas_dia in tasas_por_fecha.items()
            for moneda_destino, tasa in tasas_dia.items()
        ]

        CONEXION_SQL.executemany(
            "INSERT OR IGNORE INTO tipos_cambio VALUES (?, ?, ?, ?)", registros
        )
        CONEXION_SQL.commit()
        _COMBINACIONES_CARGADAS.add(clave)
        return True, None

    except Exception as e:
        return False, f"{type(e).__name__}: {e}"


print("Funcion de consulta a la API lista (Frankfurter -- datos del Banco Central Europeo).")


Funcion de consulta a la API lista (Frankfurter -- datos del Banco Central Europeo).


In [4]:
# Controles interactivos (ipywidgets) y generador de consultas SQL

MONEDAS_DISPONIBLES = ["MXN", "EUR", "GBP", "JPY", "CAD", "CHF", "CNY", "BRL"]

selector_base = widgets.Dropdown(
    options=["USD", "EUR", "MXN"], value="USD", description="Moneda base:",
    style={"description_width": "initial"},
)

selector_destino = widgets.SelectMultiple(
    options=MONEDAS_DISPONIBLES, value=("MXN",), description="Monedas destino:",
    rows=6, style={"description_width": "initial"}, layout=widgets.Layout(width="320px"),
)

selector_dias = widgets.Dropdown(
    options=[("Ultimos 30 dias", 30), ("Ultimos 90 dias", 90), ("Ultimos 180 dias", 180)],
    value=90, description="Rango:",
)

selector_analisis = widgets.ToggleButtons(
    options=[
        ("Serie simple", "simple"),
        ("Media movil (SQL window)", "media_movil"),
        ("Volatilidad diaria (SQL LAG)", "volatilidad"),
    ],
    description="Analisis SQL:", style={"description_width": "initial"},
)

slider_ventana = widgets.IntSlider(
    value=7, min=3, max=21, step=1, description="Ventana media movil (dias):",
    style={"description_width": "initial"}, layout=widgets.Layout(width="380px"),
)

mostrar_sql = widgets.Checkbox(value=True, description="Mostrar consulta SQL ejecutada")


def construir_consulta(moneda_base, monedas_destino, fecha_inicio, tipo_analisis, ventana):
    lista_monedas = "', '".join(monedas_destino)
    filtro_comun = (
        f"WHERE moneda_base = '{moneda_base}'\n"
        f"  AND moneda_destino IN ('{lista_monedas}')\n"
        f"  AND fecha >= '{fecha_inicio}'"
    )

    if tipo_analisis == "simple":
        return (
            "SELECT fecha, moneda_destino, tasa\n"
            "FROM tipos_cambio\n"
            f"{filtro_comun}\n"
            "ORDER BY moneda_destino, fecha;"
        )

    if tipo_analisis == "media_movil":
        return (
            "SELECT\n"
            "    fecha, moneda_destino, tasa,\n"
            "    AVG(tasa) OVER (\n"
            "        PARTITION BY moneda_destino ORDER BY fecha\n"
            f"        ROWS BETWEEN {ventana - 1} PRECEDING AND CURRENT ROW\n"
            "    ) AS media_movil\n"
            "FROM tipos_cambio\n"
            f"{filtro_comun}\n"
            "ORDER BY moneda_destino, fecha;"
        )

    # volatilidad diaria
    return (
        "SELECT\n"
        "    fecha, moneda_destino, tasa,\n"
        "    ROUND(100.0 * (tasa - LAG(tasa) OVER (PARTITION BY moneda_destino ORDER BY fecha))\n"
        "          / LAG(tasa) OVER (PARTITION BY moneda_destino ORDER BY fecha), 4) AS variacion_pct\n"
        "FROM tipos_cambio\n"
        f"{filtro_comun}\n"
        "ORDER BY moneda_destino, fecha;"
    )


print("Controles y generador de consultas SQL listos.")


Controles y generador de consultas SQL listos.


In [5]:
# Funcion principal del dashboard -- API -> SQLite -> consulta SQL -> Plotly

def actualizar_dashboard(moneda_base, monedas_destino, dias, tipo_analisis, ventana, ver_sql):
    if len(monedas_destino) == 0:
        display(Markdown("Selecciona al menos una moneda destino."))
        return

    fecha_fin = date.today()
    fecha_inicio = fecha_fin - timedelta(days=dias)

    exito, error = cargar_tipos_cambio(
        moneda_base, monedas_destino, fecha_inicio.isoformat(), fecha_fin.isoformat()
    )
    if not exito:
        mensaje_error = (
            "### No fue posible consultar la API de Frankfurter\n\n"
            "```\n" + error + "\n```\n\n"
            "Verifica tu conexion a internet o intenta con otro rango de fechas."
        )
        display(Markdown(mensaje_error))
        return

    consulta = construir_consulta(
        moneda_base, monedas_destino, fecha_inicio.isoformat(), tipo_analisis, ventana
    )

    try:
        resultado = pd.read_sql_query(consulta, CONEXION_SQL)
    except Exception as e:
        display(Markdown(f"### Error al ejecutar la consulta SQL\n\n```\n{e}\n```"))
        return

    if ver_sql:
        display(Markdown(f"**Consulta SQL ejecutada:**\n```sql\n{consulta}\n```"))

    columna_y = {"simple": "tasa", "media_movil": "media_movil", "volatilidad": "variacion_pct"}[tipo_analisis]
    titulo_y = {
        "simple": f"Tasa ({moneda_base} -> destino)",
        "media_movil": f"Media movil {ventana} dias",
        "volatilidad": "Variacion diaria (%)",
    }[tipo_analisis]

    fig = go.Figure()
    for moneda in monedas_destino:
        sub = resultado[resultado["moneda_destino"] == moneda]
        fig.add_trace(go.Scatter(
            x=sub["fecha"], y=sub[columna_y], mode="lines", name=f"{moneda_base}/{moneda}",
        ))

    fig.update_layout(
        title=f"Tipos de cambio -- {moneda_base} vs. monedas seleccionadas",
        xaxis_title="Fecha", yaxis_title=titulo_y,
        template="plotly_white", height=480, hovermode="x unified",
        margin=dict(t=70, b=40),
    )
    fig.show()


panel_interactivo = widgets.interactive(
    actualizar_dashboard,
    moneda_base=selector_base,
    monedas_destino=selector_destino,
    dias=selector_dias,
    tipo_analisis=selector_analisis,
    ventana=slider_ventana,
    ver_sql=mostrar_sql,
)

print("Panel interactivo construido. Ejecuta la siguiente celda para mostrarlo.")


Panel interactivo construido. Ejecuta la siguiente celda para mostrarlo.


In [6]:
# Renderizado del dashboard completo

display(Markdown("## Panel interactivo -- cada cambio ejecuta una consulta SQL sobre SQLite"))
display(panel_interactivo)


## Panel interactivo -- cada cambio ejecuta una consulta SQL sobre SQLite

interactive(children=(Dropdown(description='Moneda base:', options=('USD', 'EUR', 'MXN'), style=DescriptionSty…